<div style="background: linear-gradient(135deg, #0f172a, #1e293b); color:white; padding:25px; border-radius:10px;
            text-align:center; font-family:'Segoe UI', sans-serif;">

  <h1 style="margin-bottom:8px;">Market Sentiment Analysis from Tweets</h1>
  <h3 style="margin-top:0; font-style:italic; font-weight:normal; color:#94a3b8;">
    Agentic AI Classification Pipeline (Extra Credit)
  </h3>

  <hr style="width:60%; border:1px solid #475569; margin:15px auto;">

  <p style="margin:5px 0; font-size:15px;">
    <b>Group Project</b> - Text Mining (2025/2026)
  </p>
  <p style="margin:0; font-size:13px; color:#cbd5e1;">
    Master in Data Science and Advanced Analytics - Nova Information Management School
  </p>
</div>

<br>

<div style="background-color:#1e293b; color:#e2e8f0; padding:15px 20px; border-left:5px solid #475569;
            border-radius:6px; font-family:'Segoe UI', sans-serif; font-size:14px;">
  <b>Notebook Description</b><br>
  Implements a <b>conversational agentic AI workflow</b> that orchestrates the classification pipeline using the three best models from this project: DeBERTa-v3 (fine-tuned, 87.8% F1), Twitter-RoBERTa (fine-tuned, 85.9% F1), and FinBERT (fine-tuned, 82.8% F1)<br><br>
  The agent compares classifier outputs and applies a three-tier decision protocol: majority-vote tiebreaking when two models disagree, and an <b>LLM-as-a-judge override</b> when all three models disagree — where the LLM reasons directly over the tweet text using its own financial language understanding.
</div>

<br>

### Agent Decision Protocol
```
Tweet
  ↓
classify_with_deberta()  ->  classify_with_roberta()
  ↓
DeBERTa == RoBERTa?
  YES -> label  (HIGH confidence)
  NO  -> classify_with_finbert()   <- tiebreaker
          ↓
        2 of 3 agree?
          YES -> label  (MEDIUM confidence)
          NO  -> LLM reads tweet and judges directly  <- LLM-as-a-judge
                  ↓
                label  (LLM-judge confidence)
```

### Contents
1. [Setup & Model Loading](#1-setup)
2. [Classifier Tools](#2-tools)
3. [Batch Evaluation Tool](#3-coordination)
4. [Building the LangChain Agent](#4-agent)
5. [Conversational Demo](#5-demo)
6. [Batch Evaluation Demo](#6-batch)

<div id="1-setup" style="background-color:#1e293b; padding:18px; border-radius:6px;">
<h2 style="color:#f8fafc; margin:0;">1. Setup &amp; Model Loading</h2>
</div>

In [1]:
# Core imports
import os, pickle, warnings, re
import numpy as np
import pandas as pd
import torch
warnings.filterwarnings('ignore')

from transformers import AutoTokenizer, AutoModelForSequenceClassification
from scipy.special import softmax

# LangChain
from langchain_core.tools import tool
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_openai import AzureChatOpenAI
from langchain_classic.agents import AgentExecutor, create_tool_calling_agent
from langchain_classic.memory import ConversationBufferMemory

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {DEVICE}')

Device: cpu


In [2]:
# Label mapping — shared across all models
# Project labels: 0=Bearish, 1=Bullish, 2=Neutral
LABEL_NAMES = {0: 'Bearish', 1: 'Bullish', 2: 'Neutral'}

# cardiffnlp/twitter-roberta-base-sentiment-latest native labels:
#   0=negative -> Bearish, 1=neutral -> Neutral, 2=positive -> Bullish
ROBERTA_MAP = {0: 0, 1: 2, 2: 1}  # native_id -> project_id

# ProsusAI/finbert native labels:
#   0=positive -> Bullish, 1=negative -> Bearish, 2=neutral -> Neutral
FINBERT_MAP = {0: 1, 1: 0, 2: 2}  # native_id -> project_id

In [3]:
# Azure OpenAI setup (same credentials as Lab 5 / Lab 6) - not commited to GitHub for security reasons
with open('../Azure_Open_AI_Key.txt', 'r', encoding='utf-8') as f:
    key = f.read().strip()

os.environ['AZURE_OPENAI_KEY'] = key
os.environ['AZURE_OPENAI_ENDPOINT'] = 'https://novaimsplayground.openai.azure.com/'

llm = AzureChatOpenAI(
    temperature=0,
    model='ChatGPT',
    azure_endpoint=os.environ['AZURE_OPENAI_ENDPOINT'],
    openai_api_type='azure',
    api_key=os.environ['AZURE_OPENAI_KEY'],
    api_version='2024-02-15-preview',
)
print('LLM ready:', llm.model_name)

LLM ready: ChatGPT


In [4]:
# ── Load Twitter-RoBERTa and FinBERT
# Tries to load fine-tuned weights (from notebook 07); falls back to pretrained.
ROBERTA_HF = 'cardiffnlp/twitter-roberta-base-sentiment-latest'
FINBERT_HF  = 'ProsusAI/finbert'
DEBERTA_HF        = 'nickmuchi/deberta-v3-base-finetuned-finance-text-classification'

FINETUNED_ROBERTA = '../results/models/finetuning/twitter_roberta_best.pt'
FINETUNED_FINBERT  = '../results/models/finetuning/finbert_best.pt'
FINETUNED_DEBERTA = '../results/models/finetuning/deberta_best.pt'

def load_classifier(hf_name, weights_path, label, pretrained_map):
    tokenizer = AutoTokenizer.from_pretrained(hf_name)
    model = AutoModelForSequenceClassification.from_pretrained(
        hf_name, num_labels=3, ignore_mismatched_sizes=True
    )
    if os.path.exists(weights_path):
        model.load_state_dict(torch.load(weights_path, map_location=DEVICE))
        print(f'  {label}: loaded FINE-TUNED weights')
        active_map = {0: 0, 1: 1, 2: 2}  # fine-tuned: already in project label space
    else:
        print(f'  {label}: fine-tuned weights not found -> using PRETRAINED')
        active_map = pretrained_map        # pretrained: native labels need remapping
    model.to(DEVICE).eval()
    return tokenizer, model, active_map

print('Loading models...')
roberta_tok, roberta_model, roberta_active_map = load_classifier(
    ROBERTA_HF, FINETUNED_ROBERTA, 'Twitter-RoBERTa', ROBERTA_MAP)
finbert_tok, finbert_model, finbert_active_map = load_classifier(
    FINBERT_HF, FINETUNED_FINBERT, 'FinBERT', FINBERT_MAP)
deberta_tok, deberta_model, deberta_active_map = load_classifier(
    DEBERTA_HF, FINETUNED_DEBERTA, 'DeBERTa-v3', {0: 0, 1: 1, 2: 2})
print('Done.')

Loading models...


Loading weights: 100%|██████████| 201/201 [00:00<00:00, 47560.37it/s]
[transformers] RobertaForSequenceClassification LOAD REPORT from: cardiffnlp/twitter-roberta-base-sentiment-latest
Key                         | Status     |  | 
----------------------------+------------+--+-
roberta.pooler.dense.bias   | UNEXPECTED |  | 
roberta.pooler.dense.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


  Twitter-RoBERTa: loaded FINE-TUNED weights


Loading weights: 100%|██████████| 201/201 [00:00<00:00, 62555.10it/s]


  FinBERT: loaded FINE-TUNED weights


Loading weights: 100%|██████████| 202/202 [00:00<00:00, 10017.85it/s]


  DeBERTa-v3: loaded FINE-TUNED weights
Done.


<div id="2-tools" style="background-color:#1e293b; padding:18px; border-radius:6px;">
<h2 style="color:#f8fafc; margin:0;">2. Classifier Tools</h2>
</div>

Each classifier is wrapped as a LangChain `@tool`. The docstring is the **contract** the agent reads to decide when and how to call each tool.

In [5]:
def _infer_transformer(tweet, tokenizer, model, label_map):
    """Run one forward pass and return (project_label_id, confidence, all_probs_dict)."""
    enc = tokenizer(tweet, truncation=True, max_length=128,
                    return_tensors='pt').to(DEVICE)
    with torch.no_grad():
        logits = model(**enc).logits[0].cpu().numpy()
    probs = softmax(logits)
    native_id = int(np.argmax(probs))
    proj_id   = label_map[native_id]
    # Remap all probabilities to project label space
    proj_probs = {LABEL_NAMES[label_map[i]]: float(probs[i]) for i in range(3)}
    return proj_id, float(probs[native_id]), proj_probs

@tool
def classify_with_roberta(tweet: str) -> str:
    """Classify the financial sentiment of a tweet using Twitter-RoBERTa.
    Returns the predicted label (Bearish/Bullish/Neutral) and confidence score.
    This is the second model (85.9% F1, 87.9% accuracy).
    Use as second opinion after DeBERTa.
    Args:
        tweet: The raw tweet text to classify.
    """
    proj_id, conf, probs = _infer_transformer(tweet, roberta_tok, roberta_model, roberta_active_map)
    label     = LABEL_NAMES[proj_id]
    probs_str = ', '.join(f'{k}: {v:.2%}' for k, v in sorted(probs.items()))
    return f'RoBERTa -> {label} (confidence: {conf:.2%}) | all probs: [{probs_str}]'


@tool
def classify_with_finbert(tweet: str) -> str:
    """Classify the financial sentiment of a tweet using FinBERT.
    Returns the predicted label (Bearish/Bullish/Neutral) and confidence score.
    This is the third model (86.8% accuracy, F1=0.828), pre-trained on financial text.
    Use ONLY as tiebreaker when DeBERTa and RoBERTa disagree.
    Args:
        tweet: The raw tweet text to classify.
    """
    proj_id, conf, probs = _infer_transformer(tweet, finbert_tok, finbert_model, finbert_active_map)
    label     = LABEL_NAMES[proj_id]
    probs_str = ', '.join(f'{k}: {v:.2%}' for k, v in sorted(probs.items()))
    return f'FinBERT -> {label} (confidence: {conf:.2%}) | all probs: [{probs_str}]'


@tool
def classify_with_deberta(tweet: str) -> str:
    """Classify the financial sentiment of a tweet using DeBERTa-v3.
    Returns the predicted label (Bearish/Bullish/Neutral) and confidence score.
    This is the strongest model (89.7% accuracy, F1=0.878), fine-tuned from a DeBERTa-v3-base
    checkpoint pre-adapted to financial text. Called first on every tweet.
    Args:
        tweet: The raw tweet text to classify.
    """
    proj_id, conf, probs = _infer_transformer(tweet, deberta_tok, deberta_model, deberta_active_map)
    label     = LABEL_NAMES[proj_id]
    probs_str = ', '.join(f'{k}: {v:.2%}' for k, v in sorted(probs.items()))
    return f'DeBERTa -> {label} (confidence: {conf:.2%}) | all probs: [{probs_str}]'

In [6]:
# Sanity check all three tools
sample = '$AAPL crushed earnings — iPhone sales up 12%, stock soaring after hours.'
print(f'Tweet: {sample}\n')
print(classify_with_roberta.invoke(sample))
print(classify_with_finbert.invoke(sample))
print(classify_with_deberta.invoke(sample))

Tweet: $AAPL crushed earnings — iPhone sales up 12%, stock soaring after hours.

RoBERTa -> Bullish (confidence: 97.91%) | all probs: [Bearish: 1.76%, Bullish: 97.91%, Neutral: 0.33%]
FinBERT -> Bullish (confidence: 96.70%) | all probs: [Bearish: 2.46%, Bullish: 96.70%, Neutral: 0.84%]
DeBERTa -> Bullish (confidence: 97.93%) | all probs: [Bearish: 1.73%, Bullish: 97.93%, Neutral: 0.34%]


<div id="3-coordination" style="background-color:#1e293b; padding:18px; border-radius:6px;">
<h2 style="color:#f8fafc; margin:0;">3. Batch Evaluation Tool</h2>
</div>

The three classifier tools above handle individual tweets. The `evaluate_batch` tool automates evaluation over a full labelled CSV — running all three models, computing per-model accuracy and F1, and reporting the inter-model agreement rate.

The **tiebreaking and coordination logic** (calling FinBERT when DeBERTa and RoBERTa disagree, applying majority vote, reporting confidence levels) is **not hardcoded in a tool**. It is part of the agent's system prompt, so the LLM reasons through it step-by-step in the ReAct loop — making it a genuine non-trivial decision rather than a hidden Python function.

In [7]:
@tool
def evaluate_batch(csv_path: str, text_col: str = 'text', label_col: str = 'label') -> str:
    """Automatically evaluate all three models on a labelled CSV dataset.
    Computes per-model accuracy, macro F1, and inter-model agreement rate.
    Also reports the ensemble (majority vote) accuracy vs individual models.
    Use this when the user asks how models compare on a dataset, or to benchmark performance.
    Args:
        csv_path: Path to the CSV file (must have text and label columns).
        text_col: Name of the text column (default: 'text').
        label_col: Name of the label column (default: 'label') — must be 0/1/2 integers.
    """
    from sklearn.metrics import accuracy_score, f1_score
    from collections import Counter

    df = pd.read_csv(csv_path)
    if label_col not in df.columns:
        return f'Error: column "{label_col}" not found. Available columns: {list(df.columns)}'

    texts  = df[text_col].tolist()
    y_true = df[label_col].tolist()
    n = len(texts)

    print(f'  Evaluating {n} tweets...')

    r_preds, f_preds, d_preds, ens_preds = [], [], [], []
    for tweet in texts:
        r_id, _, _ = _infer_transformer(tweet, roberta_tok, roberta_model, roberta_active_map)
        f_id, _, _ = _infer_transformer(tweet, finbert_tok, finbert_model, finbert_active_map)
        d_id, _, _ = _infer_transformer(tweet, deberta_tok, deberta_model, deberta_active_map)
        maj  = Counter([r_id, f_id, d_id]).most_common(1)[0][0]
        r_preds.append(r_id); f_preds.append(f_id)
        d_preds.append(d_id); ens_preds.append(maj)

    agreement = sum(r == f == d for r, f, d in zip(r_preds, f_preds, d_preds)) / n

    rows = []
    for name, preds in [('RoBERTa', r_preds), ('FinBERT', f_preds),
                        ('DeBERTa', d_preds), ('Ensemble (majority vote)', ens_preds)]:
        acc = accuracy_score(y_true, preds)
        f1  = f1_score(y_true, preds, average='macro', zero_division=0)
        rows.append(f'  {name:<26} Acc={acc:.4f}  F1={f1:.4f}')

    return (
        f'Batch evaluation on {n} tweets from "{os.path.basename(csv_path)}"\n'
        + '\n'.join(rows)
        + f'\n\n  Inter-model agreement (all 3 same): {agreement:.2%} of tweets'
    )

In [8]:
# Sanity check evaluate_batch tool
with open('../data/processed_splits.pkl', 'rb') as f:
    splits = pickle.load(f)

sample_df = pd.DataFrame({'text': splits['x_val'][:20], 'label': splits['y_val'][:20]})
sample_path = '../data/val_sample_20.csv'
sample_df.to_csv(sample_path, index=False)

print(evaluate_batch.invoke({'csv_path': sample_path}))

  Evaluating 20 tweets...
Batch evaluation on 20 tweets from "val_sample_20.csv"
  RoBERTa                    Acc=0.8500  F1=0.8422
  FinBERT                    Acc=0.9000  F1=0.8876
  DeBERTa                    Acc=0.8500  F1=0.8422
  Ensemble (majority vote)   Acc=0.9000  F1=0.8876

  Inter-model agreement (all 3 same): 80.00% of tweets


<div id="4-agent" style="background-color:#1e293b; padding:18px; border-radius:6px;">
<h2 style="color:#f8fafc; margin:0;">4. Building the LangChain Agent</h2>
</div>

The agent uses the same **Azure OpenAI + LangChain** stack as Lab 6. It receives a user prompt in natural language and decides which tool(s) to call and in what order.

In [9]:
SYSTEM_PROMPT = """You are a financial tweet sentiment analyst with access to three classification models.

TOOLS AVAILABLE:
1. classify_with_deberta   — primary model (87.8% F1), DeBERTa-v3 fine-tuned on financial text.
2. classify_with_roberta   — second model (85.9% F1), Twitter-RoBERTa fine-tuned. Use as second opinion.
3. classify_with_finbert   — third model (82.8% F1), FinBERT fine-tuned. Use ONLY as tiebreaker.
4. evaluate_batch          — runs all three models on a full labelled CSV and reports accuracy + agreement rate.

DECISION PROTOCOL — follow this exact reasoning for every tweet classification:

  Step 1: Call classify_with_deberta. Note the label and confidence.
  Step 2: Call classify_with_roberta. Note the label and confidence.

  Step 3a — AGREEMENT: 
    Return that label with HIGH confidence. Do NOT call FinBERT.

  Step 3b — DISAGREEMENT (models predict different labels):
    Call classify_with_finbert as a tiebreaker.
    - If 2 of 3 models agree -> return the majority label with MEDIUM confidence.
    - If all 3 disagree -> go to Step 3c.

  Step 3c — LLM JUDGE (all three models disagree, no majority possible):
    Do NOT call any more tools. Instead, read the tweet text carefully yourself and reason
    about its financial sentiment using your own knowledge. Consider:
      - Key financial terms (earnings, downgrade, surge, collapse, holds steady, etc.)
      - Directional language (up, down, cuts, raises, beats, misses)
      - Overall tone and context
    Make a final judgment and label it: LLM-as-a-judge override.
    Explain which linguistic signals led to your decision.

  Step 4: Always report: final label, confidence level (HIGH / MEDIUM / LLM-judge), and reasoning.

LABELS: Bearish (negative market outlook), Bullish (positive outlook), Neutral.
"""

tools = [classify_with_deberta, classify_with_roberta, classify_with_finbert, evaluate_batch]
print('Tools registered:', [t.name for t in tools])

Tools registered: ['classify_with_deberta', 'classify_with_roberta', 'classify_with_finbert', 'evaluate_batch']


In [10]:
memory = ConversationBufferMemory(memory_key='chat_history', return_messages=True)

prompt = ChatPromptTemplate.from_messages([
    ('system', SYSTEM_PROMPT),
    MessagesPlaceholder('chat_history', optional=True),
    ('human', '{input}'),
    MessagesPlaceholder('agent_scratchpad'),
])

agent = create_tool_calling_agent(llm, tools, prompt)
agent_executor = AgentExecutor(
    agent=agent,
    tools=tools,
    verbose=True,
    memory=memory,
    return_intermediate_steps=True,
)

def chat(user_input: str):
    """Send a message and print the agent's response with its reasoning trace."""
    print(f'\n{"="*70}')
    print(f'User: {user_input}')
    print('='*70)
    result = agent_executor.invoke({'input': user_input})
    print(f'\nAgent: {result["output"]}')
    return result

print('Agent ready. Use chat("your message") to interact.')

Agent ready. Use chat("your message") to interact.


/var/folders/kx/33c35mzs1g95d1knp21wl0480000gn/T/ipykernel_64351/1786024847.py:1: LangChainDeprecationWarning: The class `ConversationBufferMemory` was deprecated in LangChain 0.3.1 and will be removed in 2.0.0. Use `langchain.agents.create_agent` instead. For agents that need to remember prior interactions, use `create_agent` with checkpointing or the `Store` API. See https://docs.langchain.com/oss/python/langchain/short-term-memory and https://docs.langchain.com/oss/python/langchain/long-term-memory
  memory = ConversationBufferMemory(memory_key='chat_history', return_messages=True)


<div id="5-demo" style="background-color:#1e293b; padding:18px; border-radius:6px;">
<h2 style="color:#f8fafc; margin:0;">5. Conversational Demo</h2>
</div>

The cells below demonstrate the agent's conversational interface and decision-making. Each cell shows a different scenario.

In [11]:
# Scenario 1: simple classification (agent calls DeBERTa then RoBERTa, both agree)
r1 = chat('Classify this tweet: "$TSLA smashes Q4 deliveries record — stock up 8% pre-market"')


User: Classify this tweet: "$TSLA smashes Q4 deliveries record — stock up 8% pre-market"


> Entering new AgentExecutor chain...

Invoking: `classify_with_deberta` with `{'tweet': '$TSLA smashes Q4 deliveries record — stock up 8% pre-market'}`


DeBERTa -> Bullish (confidence: 97.87%) | all probs: [Bearish: 1.70%, Bullish: 97.87%, Neutral: 0.44%]
Invoking: `classify_with_roberta` with `{'tweet': '$TSLA smashes Q4 deliveries record — stock up 8% pre-market'}`


RoBERTa -> Bullish (confidence: 97.92%) | all probs: [Bearish: 1.73%, Bullish: 97.92%, Neutral: 0.35%]Final label: Bullish
Confidence level: HIGH
Reasoning: Both the DeBERTa and RoBERTa models strongly agree that the tweet is Bullish with very high confidence (~98%). The tweet highlights a positive financial event ("smashes Q4 deliveries record") and a significant stock price increase ("stock up 8% pre-market"), which are clear bullish signals indicating positive market outlook for $TSLA. Since both top models agree, no tiebreak

In [12]:
# ── Scenario 2: user asks for a second opinion (agent calls both models)
r2 = chat('Can you also check with FinBERT and tell me if both models agree?')


User: Can you also check with FinBERT and tell me if both models agree?


> Entering new AgentExecutor chain...

Invoking: `classify_with_deberta` with `{'tweet': '$TSLA smashes Q4 deliveries record — stock up 8% pre-market'}`


DeBERTa -> Bullish (confidence: 97.87%) | all probs: [Bearish: 1.70%, Bullish: 97.87%, Neutral: 0.44%]
Invoking: `classify_with_roberta` with `{'tweet': '$TSLA smashes Q4 deliveries record — stock up 8% pre-market'}`


RoBERTa -> Bullish (confidence: 97.92%) | all probs: [Bearish: 1.73%, Bullish: 97.92%, Neutral: 0.35%]
Invoking: `classify_with_finbert` with `{'tweet': '$TSLA smashes Q4 deliveries record — stock up 8% pre-market'}`


FinBERT -> Bullish (confidence: 97.75%) | all probs: [Bearish: 1.72%, Bullish: 97.75%, Neutral: 0.53%]Both DeBERTa and RoBERTa models agree on the label Bullish with very high confidence (around 97.9%). FinBERT also predicts Bullish with similarly high confidence (97.75%). 

Since the top two models agree, the final classification

In [13]:
# ── Scenario 3: ambiguous tweet - agent calls all three models, FinBERT breaks the tie or LLM judges
r3 = chat(
    'Use your most reliable method to classify this ambiguous tweet: '
    '"$GS — Goldman revises outlook amid mixed macro signals; bond desk cautious but equities team sees opportunity"'
)


User: Use your most reliable method to classify this ambiguous tweet: "$GS — Goldman revises outlook amid mixed macro signals; bond desk cautious but equities team sees opportunity"


> Entering new AgentExecutor chain...

Invoking: `classify_with_deberta` with `{'tweet': '$GS — Goldman revises outlook amid mixed macro signals; bond desk cautious but equities team sees opportunity'}`


DeBERTa -> Neutral (confidence: 90.67%) | all probs: [Bearish: 6.07%, Bullish: 3.26%, Neutral: 90.67%]
Invoking: `classify_with_roberta` with `{'tweet': '$GS — Goldman revises outlook amid mixed macro signals; bond desk cautious but equities team sees opportunity'}`


RoBERTa -> Bullish (confidence: 77.10%) | all probs: [Bearish: 6.13%, Bullish: 77.10%, Neutral: 16.77%]
Invoking: `classify_with_finbert` with `{'tweet': '$GS — Goldman revises outlook amid mixed macro signals; bond desk cautious but equities team sees opportunity'}`


FinBERT -> Bearish (confidence: 57.73%) | all probs: [Bearish: 57.73%, 

In [14]:
# ── Scenario 4: batch of tweets in one message
r4 = chat(
    'Classify each of these tweets individually and tell me which is most bullish:\n'
    '1. "$AMZN raises Prime prices — margin expansion expected, analysts upgrade"\n'
    '2. "Fed holds rates steady — markets breathe sigh of relief"\n'
    '3. "$META ad revenue collapses — Q2 guidance slashed by 20%"'
)


User: Classify each of these tweets individually and tell me which is most bullish:
1. "$AMZN raises Prime prices — margin expansion expected, analysts upgrade"
2. "Fed holds rates steady — markets breathe sigh of relief"
3. "$META ad revenue collapses — Q2 guidance slashed by 20%"


> Entering new AgentExecutor chain...

Invoking: `classify_with_deberta` with `{'tweet': '$AMZN raises Prime prices — margin expansion expected, analysts upgrade'}`


DeBERTa -> Bullish (confidence: 97.72%) | all probs: [Bearish: 1.76%, Bullish: 97.72%, Neutral: 0.51%]
Invoking: `classify_with_roberta` with `{'tweet': '$AMZN raises Prime prices — margin expansion expected, analysts upgrade'}`


RoBERTa -> Bullish (confidence: 97.82%) | all probs: [Bearish: 1.84%, Bullish: 97.82%, Neutral: 0.34%]
Invoking: `classify_with_deberta` with `{'tweet': 'Fed holds rates steady — markets breathe sigh of relief'}`


DeBERTa -> Bullish (confidence: 97.55%) | all probs: [Bearish: 1.83%, Bullish: 97.55%, Neutral: 0.61%

In [15]:
# ── Scenario 5: memory — refers back to a previous classification
r5 = chat('Of all the tweets you classified so far, which one are you most uncertain about and why?')


User: Of all the tweets you classified so far, which one are you most uncertain about and why?


> Entering new AgentExecutor chain...
The tweet I am most uncertain about is:

"$GS — Goldman revises outlook amid mixed macro signals; bond desk cautious but equities team sees opportunity"

Reason for uncertainty:

- All three models disagreed on the label: DeBERTa predicted Neutral, RoBERTa predicted Bullish, and FinBERT predicted Bearish.
- This lack of consensus indicates ambiguity in the tweet's sentiment.
- The tweet contains mixed signals: cautiousness from the bond desk (bearish) versus opportunity seen by the equities team (bullish).
- The overall tone is balanced and nuanced, making it difficult for models to confidently assign a clear positive or negative sentiment.
- Therefore, I had to rely on my own judgment as LLM-as-a-judge and assign a Neutral label with LLM-judge confidence, reflecting the inherent uncertainty.

> Finished chain.

Agent: The tweet I am most uncertain abo

<div id="6-batch" style="background-color:#1e293b; padding:18px; border-radius:6px;">
<h2 style="color:#f8fafc; margin:0;">6. Batch Evaluation Demo</h2>
</div>

The agent can be prompted to evaluate a full labelled dataset — automating what would otherwise require running three separate notebooks.

In [16]:
# Reset memory for a fresh session
memory.clear()

# ── Automated evaluation: ask the agent to benchmark models on the validation set
# NOTE: evaluate_batch is slow on large datasets (runs 3 forward passes per tweet).
# For a quick demo, we first create a small sample file.
with open('../data/processed_splits.pkl', 'rb') as f:
    splits = pickle.load(f)

sample_df = pd.DataFrame({'text': splits['x_val'][:100], 'label': splits['y_val'][:100]})
sample_path = '../data/val_sample_100.csv'
sample_df.to_csv(sample_path, index=False)
print(f'Saved 100-tweet sample to {sample_path}')

Saved 100-tweet sample to ../data/val_sample_100.csv


In [17]:
r6 = chat(
    f'Evaluate all three models on the validation sample at "{sample_path}". '
    f'Tell me which model performs best and how often they agree with each other.'
)


User: Evaluate all three models on the validation sample at "../data/val_sample_100.csv". Tell me which model performs best and how often they agree with each other.


> Entering new AgentExecutor chain...

Invoking: `evaluate_batch` with `{'csv_path': '../data/val_sample_100.csv'}`


  Evaluating 100 tweets...
Batch evaluation on 100 tweets from "val_sample_100.csv"
  RoBERTa                    Acc=0.9100  F1=0.8854
  FinBERT                    Acc=0.9000  F1=0.8823
  DeBERTa                    Acc=0.8900  F1=0.8793
  Ensemble (majority vote)   Acc=0.9400  F1=0.9254

  Inter-model agreement (all 3 same): 82.00% of tweetsOn the validation sample of 100 tweets:

- RoBERTa performed best with 91.0% accuracy and F1 score of 0.8854.
- FinBERT was close behind with 90.0% accuracy and F1 of 0.8823.
- DeBERTa had 89.0% accuracy and F1 of 0.8793.

The ensemble majority vote of all three models achieved the highest accuracy of 94.0% and F1 of 0.9254.

All three models agreed on the same label 

In [18]:
# ── Follow-up: agent uses memory of evaluation results to answer a derived question
r7 = chat(
    'Based on those results, when the models disagree (the other ~X%), '
    'which model should I trust more as a tiebreaker — FinBERT or RoBERTa? Explain your reasoning.'
)


User: Based on those results, when the models disagree (the other ~X%), which model should I trust more as a tiebreaker — FinBERT or RoBERTa? Explain your reasoning.


> Entering new AgentExecutor chain...
The models agree on 82% of tweets, so they disagree on about 18% of tweets.

Between FinBERT and RoBERTa as tiebreakers when DeBERTa and RoBERTa disagree, RoBERTa is the better choice. This is because:

- RoBERTa has the highest individual accuracy (91.0%) and F1 (0.8854) on the validation sample, outperforming FinBERT (90.0% accuracy, 0.8823 F1).
- RoBERTa's stronger performance indicates it is more reliable in challenging cases where models disagree.
- FinBERT is still strong but slightly behind RoBERTa in accuracy and F1, so it is better to trust RoBERTa as the tiebreaker.

Therefore, when DeBERTa and RoBERTa disagree, use RoBERTa's prediction as the tiebreaker rather than FinBERT. This will likely yield better overall classification quality.

> Finished chain.

Agent: The models

---

## Summary: Why This Qualifies as an Agentic Workflow

| Criterion | Implementation |
|-----------|----------------|
| **Conversational interface** | `AgentExecutor` with `ConversationBufferMemory` — multi-turn, context-aware |
| **Choosing between alternative models** | Agent decides which models to call based on prior observations |
| **Comparing outputs from multiple classifiers** | Agent reads DeBERTa + RoBERTa outputs and explicitly reasons about agreement |
| **Routing tweets to different models** | FinBERT only invoked by the agent when the two primary models disagree |
| **LLM-as-a-judge** | When all 3 models disagree, the LLM reasons directly over the tweet text and overrides |
| **Automating evaluation** | `evaluate_batch` runs all three models on a full CSV and reports accuracy + agreement |

### Three-tier decision architecture

```
Tier 1 — AGREEMENT        DeBERTa == RoBERTa
                           -> return label  (HIGH confidence)
                           -> FinBERT never called

Tier 2 — MAJORITY VOTE    DeBERTa ≠ RoBERTa, FinBERT casts deciding vote
                           -> 2/3 majority wins  (MEDIUM confidence)

Tier 3 — LLM-AS-A-JUDGE   All three models disagree, no majority possible
                           -> LLM reads tweet text directly, applies financial
                             language understanding, explains reasoning
                           -> (LLM-judge confidence)
```

Tier 3 is what makes this a genuine **LLM-as-a-judge** system: the LLM does not delegate to another tool — it evaluates the ambiguous case itself, using its own knowledge as the final arbiter. The ReAct trace shows the explicit reasoning chain for every decision.